In [1]:
from transformers import AutoTokenizer, AutoProcessor
from qwen_ivtlr import IVTLR  
from transformers import Qwen2VLForConditionalGeneration
import torch
import deepspeed
from peft import LoraConfig,get_peft_model
from qwen_vl_utils import process_vision_info
from datasets import load_dataset
import re
import logging
import json
import os
import time
from datetime import timedelta
import argparse
import yaml
import pdb
import sys

/home/csalt/anaconda3/envs/ivtlr/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-05-17 20:04:28,779] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/csalt/anaconda3/envs/ivtlr/bin/../libexec/gcc/x86_64-conda-linux-gnu/14.3.0/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/csalt/anaconda3/envs/ivtlr/bin/../libexec/gcc/x86_64-conda-linux-gnu/14.3.0/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
def load_inference_model(checkpoint_path, model_name):
    processor = AutoProcessor.from_pretrained(model_name)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False,
        trust_remote_code=True,
        padding_side="right"
    )

    tokenizer.add_special_tokens({
        "additional_special_tokens": [
            "<|start-latent|>",
            "<|end-latent|>",
            "<|latent|>"
        ]
    })

    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="eager"
)

    base_model.resize_token_embeddings(len(tokenizer))
    processor.tokenizer = tokenizer

    lora_config = LoraConfig(
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        r=64,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        inference_mode=False
    )

    model = get_peft_model(base_model, lora_config)
    model.state_dict().keys()
    state_dict = torch.load(checkpoint_path, map_location="cpu")

    clean_state_dict = {}
    for k, v in state_dict.items():
        new_k = k

        if new_k.startswith("module."):
            new_k = new_k[len("module."):]

        if new_k.startswith("base_causallm."):
            new_k = new_k[len("base_causallm."):]

        clean_state_dict[new_k] = v

    missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)

    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))
    print("First missing keys:", missing[:20])
    print("First unexpected keys:", unexpected[:20])

    model = model.to(device)
    model.eval()

    return model, processor, tokenizer

In [4]:
ckpt = "/home/csalt/Haider/DVLM/IVT-LR/qwen_vl/output/qwen_IVTLR_m3cot/epoch_16_full_model_fp32.pth"
model, processor, tokenizer = load_inference_model(ckpt, "Qwen/Qwen2-VL-2B-Instruct")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.42it/s]


Missing keys: 0
Unexpected keys: 1
First missing keys: []
First unexpected keys: ['embedding.weight']


In [11]:
state_dict = torch.load(ckpt, map_location="cpu")
for i, k in enumerate(state_dict.keys()):
    print(k)
    if i > 50:
        break
    


base_causallm.base_model.model.visual.patch_embed.proj.weight
base_causallm.base_model.model.visual.blocks.0.norm1.weight
base_causallm.base_model.model.visual.blocks.0.norm1.bias
base_causallm.base_model.model.visual.blocks.0.norm2.weight
base_causallm.base_model.model.visual.blocks.0.norm2.bias
base_causallm.base_model.model.visual.blocks.0.attn.qkv.weight
base_causallm.base_model.model.visual.blocks.0.attn.qkv.bias
base_causallm.base_model.model.visual.blocks.0.attn.proj.weight
base_causallm.base_model.model.visual.blocks.0.attn.proj.bias
base_causallm.base_model.model.visual.blocks.0.mlp.fc1.weight
base_causallm.base_model.model.visual.blocks.0.mlp.fc1.bias
base_causallm.base_model.model.visual.blocks.0.mlp.fc2.weight
base_causallm.base_model.model.visual.blocks.0.mlp.fc2.bias
base_causallm.base_model.model.visual.blocks.1.norm1.weight
base_causallm.base_model.model.visual.blocks.1.norm1.bias
base_causallm.base_model.model.visual.blocks.1.norm2.weight
base_causallm.base_model.model

In [12]:
# Inspect where embedding.weight comes from in the checkpoint
target_key = "embedding.weight"
print("Has key:", target_key in state_dict)
if target_key in state_dict:
    w = state_dict[target_key]
    print("checkpoint", target_key, "shape:", tuple(w.shape), "dtype:", w.dtype)

# Show nearby embedding-like keys to identify the source module
embed_keys = [k for k in state_dict.keys() if "embed" in k or "embedding" in k]
print("embedding-like keys (first 20):")
for k in embed_keys[:20]:
    print(" ", k)

Has key: True
checkpoint embedding.weight shape: (151660, 1536) dtype: torch.bfloat16
embedding-like keys (first 20):
  base_causallm.base_model.model.visual.patch_embed.proj.weight
  base_causallm.base_model.model.model.embed_tokens.weight
  embedding.weight
